# 本地 WSL 环境与自建绿色包（选看）

**这一节是选看内容，主线（03.01~03.08）不需要它。**

主线走云端：下载 82 MB 绿色包，点几下就出结果。
但如果你想长期在自己的电脑上折腾——改源码、改 Checker 规则、改模拟行为、重新打包——
那就需要本地环境：在 Windows 里装 WSL，装 CANN，把 HCCL-VM 编出来。

这一节把这套流程完整走一遍，**耗时约 1 小时**。

## 本节学习大纲

- 在 Windows 中安装和使用 Ubuntu（WSL）
- 安装 CANN
- 编译 HCCL-VM（注意是两遍构建）
- 编译 hccl_test
- 改源码后重新部署
- 改码练习：改一条任务依赖，读 Checker 的定位信息
- 打包成绿色包

> 云端和本地的关系：本地"造"包，云端"用"包。绿色包就是在本地产出、再放到仓库里给云端用的。

## 1. 在 Windows 中安装和使用 Ubuntu

以下内容来自原课件。**注意：这些代码块是示例，不是在本 notebook 里运行的**——
带 `wsl` 的在 Windows 的 PowerShell 里执行，`cat`/`cd` 这类在 WSL 的 Ubuntu 终端里执行。

### 本节学习大纲

- Ubuntu 是什么，应该从哪里下载
- 安装 WSL 与 Ubuntu 24.04（含创建用户名和密码）
- 检查安装是否符合要求
- 在 Ubuntu 中创建工作目录
- Windows 和 Ubuntu 的文件路径如何对应
- 安装 Ubuntu 时遇到问题，先看这里
- 安装 HCCL-VM：两种路径，任选一种

### 1.1 Ubuntu 是什么，应该从哪里下载

Ubuntu 是一种 Linux 操作系统，后续的 CANN、HCCL-VM 编译和运行命令都在它里面执行。**WSL 2** 是 Windows 提供的 Linux 运行环境，让你在当前 Windows 电脑上打开、使用 Ubuntu。

本文使用微软提供的安装命令，命令会自动下载 Ubuntu。官方入口是[微软 WSL 安装说明](https://learn.microsoft.com/zh-cn/windows/wsl/install)和[Ubuntu 官方 WSL 安装说明](https://ubuntu.com/wsl/docs/latest/howto/install-ubuntu-wsl2/)，不需要先寻找 Ubuntu 桌面版的 ISO 安装文件。

开始前检查：

- **Windows 版本**：按 `Win + R`，输入 `winver`，按回车；使用 Windows 11，或 Windows 10 版本 2004、内部版本 19041 及以上。
- **处理器架构**：在“设置 → 系统 → 关于”查看“系统类型”，应为基于 x64 的处理器。HCCL-VM 需要 **x86_64** 架构，ARM 电脑不属于本文的环境基线。
- **虚拟化**：按 `Ctrl + Shift + Esc` 打开任务管理器，在“性能 → CPU”查看“虚拟化”，应显示“已启用”。若未启用，需要按电脑型号开启 BIOS/UEFI 中的虚拟化，见[微软操作说明](https://support.microsoft.com/en-us/windows/experience/enable-virtualization-on-windows)。
- **联网与权限**：下载需要联网；安装 WSL 需要 Windows 管理员权限。

### 1.2 安装 WSL 与 Ubuntu 24.04

**执行位置：Windows PowerShell，以管理员身份打开。**

1. 点击 Windows 开始菜单，搜索 `PowerShell`。
2. 右键点击“Windows PowerShell”，选择“以管理员身份运行”。
3. 若弹出系统授权窗口，点击“是”。
4. 在打开的窗口中执行下面的命令。

**动手前先确认磁盘和内存够用**，避免装到一半空间不足。

| 项目 | 需要预留 |
| --- | --- |
| CANN 安装目录 | **≥ 5 GB**（实际约 4.3 GB）|
| CANN 下载缓存 | **≥ 2 GB**（实际约 1.5 GB）|
| 源码与编译工作目录 | **≥ 10 GB** |
| WSL 与 Ubuntu 本体 | 数 GB |

**建议整机预留 20 GB 以上可用空间。**

内存方面：后面编译源码会**全核并行**，可按「**2 GB 基础 + 每核 0.5 GB**」估算
（实测 24 核满负荷编译峰值约 13 GB）。内存不足时，编译会中途被系统杀掉，
典型报错是 `Killed signal terminated program cc1plus`。**建议给 WSL 至少 8 GB 内存。**

WSL 默认会限制可用内存。若机器内存较小，可在 Windows 用户目录下新建或编辑 `.wslconfig`：

```ini
[wsl2]
memory=8GB
swap=8GB
```

改完执行 `wsl --shutdown` 重启 WSL 生效。

**第一步：下载并安装 WSL。**

```powershell
wsl --install --no-distribution
```

这条命令会下载、安装 WSL，并启用所需的 Windows 功能。`--no-distribution` 表示这一步先不安装
Ubuntu 等 Linux 系统。WSL 由命令自动获取，无需手动寻找安装包。

如果提示重启，先保存正在做的工作，再重启电脑。重启后重新打开管理员 PowerShell，继续第二步。

**第二步：安装 Ubuntu 24.04，并按提示创建账户。**

```powershell
wsl --install -d Ubuntu-24.04
```

`-d Ubuntu-24.04` 指定要安装的 Ubuntu 版本。这条命令会**一次走完安装和首次启动**，
最后要求你创建一个 Ubuntu 账户 —— 实际运行效果如下：

![安装 Ubuntu 24.04 并创建账户](./images/03.03_02_install_ubuntu.png)

*图 1 · 实际运行效果：从下载安装、首次启动，到创建用户名和密码，一条命令走完。*

安装过程中会依次要求：

- `Create a default Unix user account:` 输入你想用的用户名；
- `New password:` 与 `Retype new password:` 设置并确认密码。

这个账户属于 Ubuntu，和 Windows 账户是分开的。**输入密码时屏幕上不会显示任何字符，
也没有星号，这是正常现象**，照常输入后回车即可。以后执行 `sudo` 时用的就是这个密码。

看到 `用户名@电脑名:~$` 这样的提示符，就说明已经进入 Ubuntu 终端，可以开始输入 Linux 命令了。
WSL 里的 Ubuntu 平时主要就通过这个命令窗口操作。

若安装过程中提示重启，先保存工作再重启；重启后 Ubuntu 若还没装完，重新打开管理员
PowerShell 执行同一条命令即可。

依据：[微软安装步骤](https://learn.microsoft.com/zh-cn/windows/wsl/install)、
[WSL 基础命令](https://learn.microsoft.com/zh-cn/windows/wsl/basic-commands)、
[Ubuntu 可安装版本与安装方式](https://ubuntu.com/wsl/docs/latest/howto/install-ubuntu-wsl2/)、
[首次账户设置说明](https://learn.microsoft.com/zh-cn/windows/wsl/setup/environment)。

### 1.3 检查安装是否符合要求

**先在 Windows PowerShell 中执行：**

```powershell
wsl --list --verbose
```

找到 Ubuntu 对应的一行，确认最后一列 `VERSION` 为 `2`。

列表里**可能还有别的发行版**（例如装了 Docker Desktop 就会出现 `docker-desktop`），**只看 Ubuntu 那一行**即可：

![wsl --list --verbose 的输出](./images/03.03_03_wsl_list_verbose.png)

*图 2 · 实际运行效果：`wsl --list --verbose` 的输出，Ubuntu 那一行的 `VERSION` 为 `2`。*

`STATE` 显示 `Running` 或 `Stopped` 都正常，分别表示正在运行或尚未启动。这里的 `VERSION 2` 是 **WSL 的版本**，不是 Ubuntu 的版本。

如果新安装的 Ubuntu 显示为 WSL 1，在 PowerShell 中执行下面的转换命令，等待完成后再检查：

```powershell
wsl --set-version Ubuntu-24.04 2
```

**再打开 Ubuntu，在 Ubuntu 终端中执行：**

> 如果 Ubuntu 窗口已经关掉，在 **Windows PowerShell** 中执行 `wsl -d Ubuntu-24.04` 就能重新打开，
> 也可以在 Windows 开始菜单里直接点击 Ubuntu。

```bash
cat /etc/os-release
uname -m
```

![查看系统版本与架构](./images/03.03_04_os_release.png)

*图 3 · 实际运行效果：`cat /etc/os-release` 给出系统版本信息，`uname -m` 给出机器架构 `x86_64`。*

前一个命令应显示 Ubuntu 24.04 的版本信息，后一个应显示 `x86_64`。
两者都对上，说明环境和本教程要求一致，可以继续。

[WSL 版本检查说明](https://learn.microsoft.com/zh-cn/windows/wsl/basic-commands)

#### 【注】命令应该输在哪里

如果安装后的 Ubuntu 名称与 `Ubuntu-24.04` 不同，用 `wsl --list --verbose` 中显示的名称替换命令里的名字。

后面会遇到三种终端：

| 当前环境 | 提示符示例 | 在这里执行什么 |
|---|---|---|
| Windows PowerShell | `PS C:\Users\admin>` | `wsl ...`，安装、启动和管理 Ubuntu |
| Ubuntu 终端 | `hccllearner@电脑名:~$` | `sudo`、`apt`、`cd`、`git`、编译脚本等 Linux 命令 |
| HCCL-VM 交互终端 | `(hvm)$>` | 模拟通信配置、`mpirun` 测试和插件校验 |

**Ubuntu 的 Bash 代码单元**：连接到 Ubuntu/WSL 中的 Bash 内核后，点击单元左侧的运行按钮，或按 `Shift + Enter`，实际输出会显示在单元下方。按顺序执行，等当前单元结束后再继续。

**Windows PowerShell 命令**：复制到 Windows PowerShell 执行。复制时不要带提示符或代码块围栏。需要把 Bash 单元中的命令复制到 Ubuntu 终端时，可用 `Ctrl + Shift + V` 粘贴。

在 Ubuntu 终端输入 `exit` 可以退出当前终端；已经安装的软件和保存的文件会保留。完成本节 1.2 节的首次安装后，下一次打开 Ubuntu，再按 3.5 节的用法跑测试。

### 1.4 在 Ubuntu 中创建工作目录

**执行位置：Ubuntu 终端。**

本文统一使用 `/home/workspace` 作为工作目录。在刚准备好的个人 Ubuntu 环境中，执行下面的命令。

**下面的单元含 `sudo`。如果需要输入密码，请把该单元的命令复制到普通 Ubuntu 终端执行，避免在 notebook 中等待密码输入。**

```bash
sudo mkdir -p /home/workspace
sudo chown "$(id -u):$(id -g)" /home/workspace
mkdir -p /home/workspace/downloads
cd /home/workspace
pwd
```

逐行解释：

- `sudo mkdir -p ...`：用管理员权限创建工作文件夹；`-p` 表示缺少上级目录时一起创建，目录已存在时不报错。
- `sudo chown ...`：把这个工作文件夹交给当前 Ubuntu 用户管理，后面下载、编译时就能写入文件。
- `mkdir -p .../downloads`：创建存放 CANN 安装包的子文件夹。
- `cd ...`：进入这个文件夹，相当于切换当前操作位置。
- `pwd`：显示当前所在位置，结果应为 `/home/workspace`。

这里的 `sudo` 表示临时使用 Ubuntu 管理员权限。若提示输入密码，输入 1.2 节创建的密码；输入时仍然不会显示字符。

如果想在 Windows 文件资源管理器中看到这个工作目录，在 Ubuntu 终端执行：

```bash
cd /home/workspace
explorer.exe .
```

第一行先进入工作目录，第二行中的 `.` 表示当前目录。它会打开 Windows 文件资源管理器，展示这个 Ubuntu 文件夹。[跨系统查看文件的方法](https://learn.microsoft.com/zh-cn/windows/wsl/filesystems)

#### 【注】把已经安装的 Ubuntu 整体迁移到 D 盘

`/home/workspace` 保存在 Ubuntu 的虚拟磁盘中。默认安装通常占用 C 盘空间；迁移整个 Ubuntu 数据磁盘后，这个 Linux 路径仍然叫 `/home/workspace`，后面的安装和编译路径可以继续照用。[WSL 虚拟磁盘说明](https://learn.microsoft.com/zh-cn/windows/wsl/disk-space)

**执行位置：Windows PowerShell。** 以下以 `Ubuntu-24.04` 和目标目录 `D:\WSL\Ubuntu-24.04` 为例，迁移到其他盘时替换目标路径即可。

先检查 Ubuntu 当前的存放位置，复制下面整行执行：

```powershell
(Get-ChildItem -Path HKCU:\Software\Microsoft\Windows\CurrentVersion\Lxss | Where-Object { $_.GetValue("DistributionName") -eq 'Ubuntu-24.04' }).GetValue("BasePath")
```

如果已经显示 `D:\WSL\Ubuntu-24.04`，跳过迁移，直接启动 Ubuntu 检查是否正常。若仍在 C 盘，先保存 Ubuntu 中的工作，并退出 Ubuntu 窗口、Docker Desktop 等使用 WSL 的程序，然后逐条执行：

```powershell
New-Item -ItemType Directory -Path "D:\WSL" -Force
wsl --shutdown
wsl --manage Ubuntu-24.04 --move "D:\WSL\Ubuntu-24.04"
```

**实际运行效果如下：**

![迁移到 D 盘](./images/03.03_05_move_to_d.png)

*图 4 · 实际运行效果：先查到 Ubuntu 在 C 盘，接着创建 D 盘目录、关闭 WSL，再执行迁移，最后提示「操作成功完成」。*

- 第一行创建 D 盘上的存放目录。
- 第二行关闭整个 WSL 运行环境，释放虚拟磁盘占用；正在运行的其他 WSL 发行版也会停止。
- 第三行把 Ubuntu 数据磁盘迁到目标目录，等待命令结束后再继续。目标子目录应使用一个新的存放位置。

迁移完成后，再次执行前面的路径检查命令，确认位置已经变为 D 盘，并启动 Ubuntu：

```powershell
wsl -d Ubuntu-24.04
```

能够正常进入 Ubuntu 后，继续使用 `/home/workspace`。在 Windows 里，D 盘目标目录主要保存 `ext4.vhdx` 虚拟磁盘文件；要查看里面的工作目录，仍使用 Ubuntu 中的 `explorer.exe .`，或在 Windows 文件资源管理器地址栏输入 `\\wsl$\Ubuntu-24.04\home\workspace`。

**如果出现 `ERROR_SHARING_VIOLATION`（另一个程序正在使用此文件）：** 先退出 Docker Desktop 等 WSL 程序，执行 `wsl --shutdown`，等待片刻。然后重新检查存放位置：若已经变为 D 盘，先尝试启动 Ubuntu 验证；若仍在 C 盘，确认目标目录没有已迁移的数据后再重试。不要因报错直接删除目标目录中的虚拟磁盘文件。

若命令提示不支持 `--manage` 或 `--move`，先执行 `wsl --update` 更新 WSL。迁移命令依据：[WSL 官方迁移功能](https://github.com/microsoft/WSL/releases/tag/2.3.11)；路径检查依据：[微软安装位置检查说明](https://learn.microsoft.com/zh-cn/windows/wsl/disk-space)。

### 1.5 Windows 和 Ubuntu 的文件路径如何对应

`/home/workspace` 是 Ubuntu 中的文件夹。Windows 磁盘在 Ubuntu 中通常通过 `/mnt/盘符/` 访问：

| Windows 路径 | 在 Ubuntu 中访问的路径 |
|---|---|
| `D:\HCCLDownloads` | `/mnt/d/HCCLDownloads` |
| `D:\HCCL课程与资料优化` | `/mnt/d/HCCL课程与资料优化` |
| `C:\Users\admin\Downloads` | `/mnt/c/Users/admin/Downloads` |

因此，可以用 Windows 浏览器下载 CANN 安装包，再从 Ubuntu 复制进去，具体操作见本章第 2 节。

本项目的源码和编译产物放在 Ubuntu 的 `/home/workspace` 下；Windows 上的课程资料仍可存放在 `D:\HCCL课程与资料优化`。微软建议使用 Linux 工具处理的项目文件放在 Linux 文件系统中，以获得更好的文件访问性能。[路径与文件存储说明](https://learn.microsoft.com/zh-cn/windows/wsl/filesystems)

### 1.6 安装 Ubuntu 时遇到问题，先看这里

如果下载停在 `0.0%` 或 Microsoft Store 下载失败，在 **管理员 PowerShell** 中尝试官方提供的另一种下载方式：

```powershell
wsl --install --web-download -d Ubuntu-24.04
```

若提示需要更新 WSL，在 PowerShell 中执行：

```powershell
wsl --update
```

如果报错 `0x80370102`，先检查 CPU 虚拟化是否开启，以及安装后是否已经重启。若提示“未安装分发版”，需要继续安装 Ubuntu，只有 WSL 组件还不能进入 Ubuntu。

仍有问题时，保留报错文字，再按具体错误处理，不必继续执行后面的 CANN 安装步骤。[微软安装说明](https://learn.microsoft.com/zh-cn/windows/wsl/install)、[微软故障排查说明](https://learn.microsoft.com/zh-cn/windows/wsl/troubleshooting)

### 1.7 安装 HCCL-VM：两种路径，任选一种

Ubuntu 准备好之后，接下来要装 HCCL-VM。**这里有两条路，选一条走完即可**，不必两条都走。

#### 路径一：一键式安装（想尽快跑起来就选它）

**执行位置：Ubuntu 终端。**

在 Ubuntu 里粘贴执行下面这一条命令：

```bash
curl -fsSL https://raw.gitcode.com/cann/hcomm/raw/master/test/hccl_vm/hccl_vm_installer | bash
```

`curl` 负责把安装脚本下载下来，`| bash` 表示下载完立刻交给 bash 执行 ——
所以**不用先保存文件**，复制整行、粘贴、回车即可。

**装到哪里？** 默认装在**你当前所在的目录**。所以最省事的做法是：

```bash
cd /home/workspace
curl -fsSL https://raw.gitcode.com/cann/hcomm/raw/master/test/hccl_vm/hccl_vm_installer | bash
```

也可以不切换目录，直接用 `--workspace` 指定：

```bash
curl -fsSL https://raw.gitcode.com/cann/hcomm/raw/master/test/hccl_vm/hccl_vm_installer | bash -s -- --workspace /home/workspace
```

**这条命令怎么看？** `bash -s` 表示「从标准输入读取脚本」，`--` 表示「后面这些是传给脚本的参数，
不是给 bash 的」—— 于是 `--workspace /home/workspace` 就被交给安装脚本了。

> ⚠️ **`--workspace` 要写 Ubuntu 里的路径，不是 Windows 路径。**
>
> 因为脚本运行在 Ubuntu 里，它看到的路径都是 Linux 格式：
>
> | 你想装的位置 | `--workspace` 该写 |
> | --- | --- |
> | Ubuntu 里的 `/home/workspace`（本教程使用）| `/home/workspace` |
> | Windows 的 `D:\hvm` | `/mnt/d/hvm` —— **不推荐**，跨文件系统编译会慢很多 |
>
> **如果你已经按前面的【注】把 Ubuntu 迁移到了 D 盘**，那么 `/home/workspace`
> **本来就在 D 盘上** ✓ —— 不需要再改成什么 Windows 路径，直接用 `/home/workspace` 就行。

**回车之后会发生什么？**

脚本会自动完成下面 9 步，**每一步都会在终端里打印进度**：

| 步骤 | 做什么 |
| --- | --- |
| 1~3 | 检查环境与磁盘空间 → 安装系统依赖 |
| 4~6 | 拉取 hcomm / hccl 源码 → 检测或下载安装 CANN |
| 7 | 编译 HCCL-VM 工具 |
| 8 | 编译安装 HCCL / HCOMM 子包 |
| 9 | 编译 `hccl_test` |

**过程中的几点预期**：

- **耗时比较长**，其中**下载和安装 CANN 最慢**（几个 GB）。中途终端可能一段时间没有新输出，
  那是在下载，**不要中断、也不要关掉窗口**；
- 脚本默认**自动安装依赖、自动确认**；遇到需要你确认的操作会给出提示，**超时则按默认值继续**；
- 如果某一步失败，脚本会打印失败原因和补救建议，按提示处理即可。

**装完的标志**：终端最后会打印一段「后续步骤」提示，里面给出可以直接复制的测试命令。
**看到它，就说明装好了。**

![一键式安装的实际运行结果](./images/03.03_06_installer_result.png)

*图 5 · 实际运行效果：一键式安装从第 1 步「检查运行环境」跑到第 9 步「编译 hccl_test」，
最后打印「集群配置示例」和可复制的后续命令 —— 看到这段就说明装好了。*

**然后直接进入 3.5 跑通第一个通信测试 即可。**

#### 路径二：分步安装（想弄清每一步在做什么就选它）

**执行位置：Ubuntu 终端**（只有「打开 Ubuntu」那一条是在 Windows PowerShell 里）。

照着本章第 2、3 节一节一节做，
每一步都自己敲命令、自己看结果。

**分步安装的好处**：

- **看清每一步在做什么** —— 系统依赖装了什么、CANN 起什么作用、源码从哪来、编译产物落在哪里，
  全都摆在眼前，而不是等脚本刷一屏日志；
- **出问题知道去哪查** —— 哪一步失败、失败在哪一环，心里有数；
- **方便按需改动** —— 换 CANN 路径、换源码分支、只想重编某一个组件时，知道该动哪个地方；
- **后续章节更顺** —— 3.6 起的通信域配置、本章的改源码与重新打包，
  都建立在分步安装时建立起来的目录结构上。

> **怎么选？** 只想先把环境跑通，走**路径一**；想把安装过程弄明白、或者后面打算改源码调试，
> 建议走**路径二**。

## 2. 安装 CANN

### 2.1 为什么对 CANN 版本有要求

镜像或老版本 CANN（如 8.5）**编不过 HCCL-VM**，会报两类硬错误：

```
fatal error: ascend_hal.h: No such file or directory      # 缺 driver 头文件
error: 'HcclCommSymWindow' has not been declared          # 缺 9.x 新增类型
```

所以要装**最新构建的 CANN**，发布在华为云 ascend devcloud 的 [cann-run-mirror](https://ascend.devcloud.huaweicloud.com/artifactory/cann-run-mirror/software/master/) 上，包名见 2.2 的表格。

> **注意区分"编"和"跑"**：这里装的 CANN 是**编译**用的。
> 打出来的绿色包自带运行库（并在二进制里写了 RPATH），**运行环境不需要装 CANN**——
> 所以在 cannlab 的 CANN 9.0.0 环境里也能跑，不会被系统版本影响。

### 2.2 两个包都要

| 包 | 大小 | 提供什么 |
| --- | --- | --- |
| `Ascend-cann-toolkit_*~weekly.*_linux-x86_64.run` | 1.44 GB | 编译器、头文件、`libhcomm`、`libruntime_v*.so`、`conf/` |
| `Ascend-cann-950-ops_*~weekly.*_linux-x86_64.run` | 3.39 GB | `libhccl.so`、`hccl.h`（在 `cann-hccl` 子包里）、`aicpu_hccl.tar.gz` |

**只装 toolkit 是不够的**：`libhccl.so` 和 `hccl.h` 属于 ops 里的 `cann-hccl` 子包。

### 2.3 安装

把两个 `.run` 放到 `~/downloads/`，然后：

```bash
cd ~/downloads
chmod +x Ascend-cann-toolkit_*~weekly.*_linux-x86_64.run
chmod +x Ascend-cann-950-ops_*~weekly.*_linux-x86_64.run

./Ascend-cann-toolkit_*~weekly.*_linux-x86_64.run --install --install-path=$HOME/Ascend --quiet
./Ascend-cann-950-ops_*~weekly.*_linux-x86_64.run --install --install-path=$HOME/Ascend --quiet

source $HOME/Ascend/cann/set_env.sh
echo "ASCEND_HOME_PATH=$ASCEND_HOME_PATH"
```

装完确认三个关键文件都在：

```bash
ls $ASCEND_HOME_PATH/include/hccl/hccl.h
ls $ASCEND_HOME_PATH/lib64/libhccl.so
find $ASCEND_HOME_PATH -name aicpu_hccl.tar.gz
```

> CANN 的头文件和库在 `<arch>-linux/lib64/` 下，同时 `lib64` 是它的软链。
> **不需要**手工给 `include/hccl` 做软链——装完 ops 就有。

## 3. 编译 HCCL-VM

### 3.1 关键：这是两遍构建

`CMakeLists.txt` 里有一段条件：

```cmake
if (NOT BUILD_DEVICE_ARM)
    config_hcclvm_install()      # install 规则在这里
    add_subdirectory(src)        # 主机侧 hccl-vm 在这里
endif()
```

也就是说：**`BUILD_DEVICE_ARM=ON` 时，主机侧和 install 规则根本不会生成**。
官方的 `build.sh --full` 其实是跑两遍：

1. `BUILD_DEVICE_ARM=OFF` → 编主机侧（`hccl-vm`、`plugin`）并 `make install`
2. `BUILD_DEVICE_ARM=ON` → 编 ARM 设备侧（`device`）

只跑第二遍，会得到"没有 install 目标、没有 `hccl-vm`"的半成品。

### 3.2 还要注意 -j 别开太大

`build.sh` 里写死 `make -j16`。8 核 16 GB 的机器会 OOM（内核被 kill）。
用 `-j6`，不行再降到 `-j3` 增量续跑。

### 3.3 开始编译

```bash
mkdir -p ~/work && cd ~/work
[ -d hccl ]  || git clone --depth 1 https://gitcode.com/cann/hccl.git
[ -d hcomm ] || git clone --depth 1 https://gitcode.com/cann/hcomm.git

source $HOME/Ascend/cann/set_env.sh
cd ~/work/hcomm/test/hccl_vm

# 第一遍：主机侧 + install
cmake -S . -B build -DBUILD_DEVICE_ARM=OFF -DBUILD_AIV=ON \
  -DASCEND_CANN_PACKAGE_PATH="$ASCEND_HOME_PATH" \
  -DHCOMM_CODE_ROOT="$HOME/work/hcomm" -DHCCL_CODE_ROOT="$HOME/work/hccl"
cd build
make -j6 || make -j3
make install -j3

# 第二遍：设备侧
cd ~/work/hcomm/test/hccl_vm
cmake -S . -B build -DBUILD_DEVICE_ARM=ON -DBUILD_AIV=ON \
  -DASCEND_CANN_PACKAGE_PATH="$ASCEND_HOME_PATH" \
  -DHCOMM_CODE_ROOT="$HOME/work/hcomm" -DHCCL_CODE_ROOT="$HOME/work/hccl"
cd build
make -j6 || make -j3
```

编完 `hccl_vm_install/` 下应有 7 个目录：

```
archive  bin  config  data  lib  plugin  script
```

### 3.4 部署设备侧依赖

`build_pkg.sh` 会把 `aicpu_hccl.tar.gz` 解出来部署到 `lib/aarch64`（x86 上是 `lib/x86_64`），
`bin/device` 就靠这些库。

```bash
cd ~/work/hcomm/test/hccl_vm
export HCOMM_CODE_HOME="$HOME/work/hcomm"
export HCCL_CODE_HOME="$HOME/work/hccl"
bash build_pkg.sh

ls hccl_vm_install/lib/*/ | head
```

## 4. 编译 hccl_test

算例来自 CANN 自带的源码，编译时要注意两件事：**补 `-lmpi_cxx`**、**`MPI_HOME` 用对架构**。

```bash
sudo apt-get update -qq
sudo apt-get install -y openmpi-bin libopenmpi-dev

source $HOME/Ascend/cann/set_env.sh
cd "$ASCEND_HOME_PATH/tools/hccl_test"

# 补 C++ 接口的库（默认只链接了 C 接口）
if ! grep -q -- '-lmpi_cxx' Makefile; then
    sed -i 's/-lmpi/-lmpi -lmpi_cxx/g' Makefile
fi

# x86 用 /usr/lib/x86_64-linux-gnu/openmpi
# aarch64 用 /usr/lib/aarch64-linux-gnu/openmpi
MPI_HOME=/usr/lib/x86_64-linux-gnu/openmpi make ASCEND_DIR="$ASCEND_HOME_PATH" -j4

ls "$ASCEND_HOME_PATH/tools/hccl_test/bin" | head
```

> 如果链接时报 `libhcomm.so: undefined reference to 'Hccl::TaskService::...'`，
> 那是 `--as-needed` 把 `-lhccl` 丢了。改成：
>
> ```
> make LIBS="-L$ASCEND_HOME_PATH/lib64 -lhcomm -lhccl -lacl_rt \
>   -L/usr/lib/x86_64-linux-gnu/openmpi/lib -lmpi -lmpi_cxx \
>   -L$ASCEND_HOME_PATH/lib64 -lmsprofiler -Wl,--allow-shlib-undefined"
> ```

## 5. 改源码后重新部署

改完 `hcomm` 或 `hccl` 的源码，要重新编译并**把产物更新到实际运行目录**，
否则跑的还是旧版本。

```bash
cd ~/work/hcomm/test/hccl_vm/build
make -j6
cd ..
bash build_pkg.sh --install hccl
bash build_pkg.sh --install hcomm
```

## 6. 改码练习：改一条任务依赖，读 Checker 的定位信息

这一段来自原课件，**必须在本地环境里做**（云端包里没有源码）。

流程是：编一个最小的任务图测试 → 跑出正常基线 → 故意改错一条依赖 →
看 Checker 怎么报 → 恢复源码再跑一次。目的是学会**从 Checker 的报错反推是哪条依赖错了**。

### 内部算法问题：缺少依赖，两个任务同时写入

算法会把通信拆成多个搬运、归约和同步任务。**任务图**用节点表示任务，用箭头表示先后依赖：`A → B` 表示 B 要等 A 完成。

下面直接调用北极星内部的 Checker 检查函数，使用仓库里的最小任务图单元测试；它独立于前面的完整 ReduceScatter 进程。单元测试只检查一个小功能，适合把“为什么出错”缩小到一处代码。

这张图中有两个搬运任务，都写入设备 0 的 CCL 缓冲区。CCL 缓冲区是算法中转数据使用的内存：

- A 写入 `[0x20000, 0x20100)`，长度为 `0x100 = 256` 字节。
- B 写入 `[0x20080, 0x20180)`，长度同样为 256 字节。
- 两个区间共同覆盖 `[0x20080, 0x20100)`，共 128 字节。

方括号表示包含起始地址，圆括号表示不包含结束地址。图中的地址是用来描述任务的示例地址。

正常情况下 A 先写、B 后写，复用这 128 字节没有问题。去掉 A 到 B 的依赖后，两个任务可以同时写，Checker 就会报告冲突。

![图 6](./images/png/03_08_dependency_conflict.png)

*图 6 · 同一段内存，有依赖时顺序复用，缺依赖时可能并发写入*

### 编译最小任务图测试，先确认正常基线

#### 4.1 准备环境与日志

这里使用仓库已有测试 `MemConflictTest.DensePathAllowsOrderedWrites`，文件位于 `test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc`。

先加载 CANN 环境并进入 HCCL-VM 源码目录。

```bash
source /home/workspace/Ascend/cann/set_env.sh
export HCOMM_CODE_HOME=/home/workspace/hcomm
export HCCL_CODE_HOME=/home/workspace/hccl
cd "$HCOMM_CODE_HOME/test/hccl_vm"
```

下面为这个测试函数开启日志，并把修改前的文件保存为同目录下的 `.tutorial39.bak` 备份。已有同名备份时命令会停止，方便保留上一次实验记录。

日志初始化让后面的错误编号、任务节点和内存区间能显示出来；这一小段准备代码没有改变任务之间的依赖。
脚本按测试函数的文本定位并做检查；如果你的仓库版本与本文不一致，脚本会停下来提示，不会改坏文件。

```bash
python3 - <<'PY'
from pathlib import Path

p = Path("test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc")
backup = p.with_name(p.name + ".tutorial39.bak")
raw = p.read_bytes()
source = raw.decode("utf-8")
marker = 'TEST_F(MemConflictTest, DensePathAllowsOrderedWrites)\n    {'
begin = source.index(marker)
end = source.index("\n    }", begin) + len("\n    }")
block = source[begin:end]
if block.count("taskA->AddChild(taskB);") != 1:
    raise SystemExit("测试函数与本节示例不一致，请保留文件并检查版本。")
if '#include <gtest/gtest.h>\n' not in source or '#include "sim_log.h"' in source:
    raise SystemExit("头文件与本节示例不一致，请保留文件并检查版本。")
block = block.replace(marker + "\n", marker + "\n" + '        if (g_logger == nullptr) {\n            InitLogger(LogConfig{});\n        }\n', 1)
source = source[:begin] + block + source[end:]
source = source.replace('#include <gtest/gtest.h>\n', '#include <gtest/gtest.h>\n#include "sim_log.h"\n', 1)
with backup.open("xb") as f:
    f.write(raw)
p.write_text(source, encoding="utf-8")
print("原文件备份：", backup)
PY
```

#### 4.2 只构建需要的测试目标

使用独立目录 `build_checker_tutorial` 保存本节构建结果。`BUILD_TESTS=ON` 开启仓库的单元测试目标；只编译内存冲突检查对应的可执行文件。

首次构建还需要 GoogleTest（C++ 单元测试框架）。仓库会优先使用 `third_party/` 中已有的依赖；缺少时自动下载、编译 GoogleTest 1.14.0，需要能够访问下载源。

```bash
cmake -S . -B build_checker_tutorial \
  -DCMAKE_BUILD_TYPE=Debug \
  -DBUILD_TESTS=ON -DBUILD_DEVICE_ARM=OFF \
  -DASCEND_CANN_PACKAGE_PATH="$ASCEND_HOME_PATH"
cmake --build build_checker_tutorial \
  --target test_task_graph_mem_conflict_v3 --parallel 4
```

运行指定的一个测试，保存基线日志。`--gtest_filter` 后的名字限定本次执行的测试函数。

```bash
./build_checker_tutorial/output/bin/test_task_graph_mem_conflict_v3 \
  --gtest_filter=MemConflictTest.DensePathAllowsOrderedWrites > baseline_task_graph.log 2>&1
printf '退出码：%s\n' "$?"
grep -n -E 'RUN|OK|FAILED|PASSED' baseline_task_graph.log
```

正常基线的退出码应为 0，测试应显示通过。下面是实跑输出：

![基线测试的实际输出](./images/03.09_baseline_test_output.png)

*图 7 · 基线测试的实际输出。`退出码：0` 表示测试程序正常结束；`[ RUN ]` 与 `[ OK ]` 行行首的 `6`、`18` 是日志文件里的行号（`grep -n` 的输出），`MemConflictTest.DensePathAllowsOrderedWrites (0 ms)` 表示这个「有先后依赖、顺序复用内存」的用例通过校验；`[ PASSED ] 1 test` 是 GoogleTest 的总结论。这就是修改前的正常基线——同样的代码、同样的检查，结论是「没有冲突」。*

### 改错一条依赖，读取 Checker 的定位信息

这个测试里，正常依赖由两行代码建立。

```cpp
start_->AddChild(taskA);
taskA->AddChild(taskB);
```

第一行把 A 接到起点，第二行把 B 接到 A 后面。现在只改第二行，让 B 也直接接到起点：

```cpp
start_->AddChild(taskA);
start_->AddChild(taskB);
```

继续在 HCCL-VM 源码目录执行下面的修改命令。它只在指定测试函数内替换这一行。

```cpp
python3 - <<'PY'
from pathlib import Path

p = Path("test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc")
source = p.read_text(encoding="utf-8")
begin = source.index('TEST_F(MemConflictTest, DensePathAllowsOrderedWrites)\n    {')
end = source.index("\n    }", begin) + len("\n    }")
block = source[begin:end]
old = "        taskA->AddChild(taskB);"
if block.count(old) != 1:
    raise SystemExit("没有找到唯一的正常依赖，文件未修改。")
block = block.replace(old, "        start_->AddChild(taskB);", 1)
p.write_text(source[:begin] + block + source[end:], encoding="utf-8")
PY
```

源码改完后需要重新编译，再运行同一个测试。

```bash
cmake --build build_checker_tutorial \
  --target test_task_graph_mem_conflict_v3 --parallel 4
./build_checker_tutorial/output/bin/test_task_graph_mem_conflict_v3 \
  --gtest_filter=MemConflictTest.DensePathAllowsOrderedWrites > missing_dependency.log 2>&1
printf '退出码：%s\n' "$?"
grep -n -A14 'ErrorCode: 302' missing_dependency.log
grep -n -E 'FAILED|PASSED' missing_dependency.log
```

这一轮预期失败，退出码应为 1。下面是实跑的完整输出——编译链接通过后运行测试，`grep` 从日志里截出了错误详情：

![缺少依赖后的编译与测试输出](./images/03.09_missing_dependency_output.png)

*图 8 · 改坏依赖后重新编译、运行测试的实际输出。`退出码：1` 表示测试失败（本轮的预期结果）。关键行是 `grep -A14` 截出的 `[ErrorCode: 302] Two tasks may access the same memory range in parallel, and at least one access is a write`（两个任务可能并发访问同一内存区间，且至少一个是写入），重叠区间 `[0x20080,0x20100)` 正是图 6 里 A、B 共同写入的 128 字节；下面的 `Conflict task 1 / 2` 给出了两个任务各自的完整访问区间和节点编号。末尾 `Expected equality of these values: CheckDataMoveTaskMemConflict(start_, &stats)` 与 `0` 是 GTest 断言失败的位置——Checker 实际返回了 `3`（HCCL_E_MEMORY），而测试仍按“正常图”预期返回 `0`；`[ FAILED ]`、`1 test, listed below` 是 GTest 的总结论。*

读这几行时，按下面的顺序定位：

1. **看错误类别。** `302` 表示并发内存访问冲突；`PHYSICAL_GLOBAL` 表示按设备上的实际地址检查这段内存。
2. **看参与设备和重叠地址。** 这里是设备 0 的 `[0x20080, 0x20100)`，正好对应图中共同写入的 128 字节。
3. **看两个冲突任务。** 后面的 `Conflict task 1`、`Conflict task 2` 会给出节点编号、读写动作和完整访问区间；这里对应搬运 A、B，即节点 1、2，排列顺序以日志为准。
4. **回到建立任务依赖的代码。** 两个任务都接在起点上，没有 A 到 B 的箭头，B 就失去了等待 A 的约束。

测试仍保留“应该成功”的断言，所以 GTest 会显示 `FAILED`：Checker 实际返回 `HCCL_E_MEMORY = 3`，而正常图的预期返回值是 `HCCL_SUCCESS = 0`。这里修复的是任务依赖，成功断言继续保留。

### 恢复源码，再跑同一个测试

下面从备份恢复原文件。只完成日志准备、尚未改错依赖时，也可以用这段命令回滚。

恢复前会比较当前内容与本节预期改动：如果文件还有其他修改，命令会停止并保留当前文件和备份，便于核对。

```cpp
python3 - <<'PY'
from pathlib import Path

p = Path("test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc")
backup = p.with_name(p.name + ".tutorial39.bak")
raw = backup.read_bytes()
expected = raw.decode("utf-8")
marker = 'TEST_F(MemConflictTest, DensePathAllowsOrderedWrites)\n    {'
begin = expected.index(marker)
end = expected.index("\n    }", begin) + len("\n    }")
block = expected[begin:end]
block = block.replace(marker + "\n", marker + "\n" + '        if (g_logger == nullptr) {\n            InitLogger(LogConfig{});\n        }\n', 1)
normal = expected[:begin] + block + expected[end:]
bad_block = block.replace("        taskA->AddChild(taskB);", "        start_->AddChild(taskB);", 1)
fault = expected[:begin] + bad_block + expected[end:]
normal = normal.replace('#include <gtest/gtest.h>\n', '#include <gtest/gtest.h>\n#include "sim_log.h"\n', 1)
fault = fault.replace('#include <gtest/gtest.h>\n', '#include <gtest/gtest.h>\n#include "sim_log.h"\n', 1)
if p.read_text(encoding="utf-8") not in (normal, fault):
    raise SystemExit("文件还有本实验之外的修改，已保留当前文件和备份；请手动核对。")
p.write_bytes(raw)
print("已恢复原文件，备份继续保留：", backup)
PY
```

恢复后重新编译，仍运行同一个测试函数。

```bash
cmake --build build_checker_tutorial \
  --target test_task_graph_mem_conflict_v3 --parallel 4
./build_checker_tutorial/output/bin/test_task_graph_mem_conflict_v3 \
  --gtest_filter=MemConflictTest.DensePathAllowsOrderedWrites > restored_task_graph.log 2>&1
printf '退出码：%s\n' "$?"
grep -n -E 'RUN|OK|FAILED|PASSED' restored_task_graph.log
```

![恢复后测试的实际输出](./images/03.09_restored_test_output.png)

*图 9 · 恢复源码后重新编译、运行同一个测试的实际输出。编译只重编了这一个测试文件（`Building CXX object ... task_graph_mem_conflict_v3_test.cc.o` 后直接链接），说明改动范围确实很小；`退出码：0`、`[ OK ]`、`[ PASSED ] 1 test` 表示测试重新通过——把 `start_->AddChild(taskB)` 改回 `taskA->AddChild(taskB)` 之后，冲突消失，与图 7 的基线结果一致。*

这次退出码应回到 0，测试重新显示 `PASSED`。保留这三份日志，就能比较同一张任务图在修改前、缺少依赖时和修复后的结果：

```bash
ls -l baseline_task_graph.log missing_dependency.log restored_task_graph.log
```

![三份日志文件](./images/03.09_three_logs_ls.png)

*图 10 · 上面命令的实际输出，三份日志都在。缺少依赖时的 `missing_dependency.log` 最大（4229 字节），多出的部分正是 302 冲突详情；另外两份记录的是通过的测试，内容简短。*

原文件备份继续保留。下次从头重复这个实验前，在 HCCL-VM 源码目录把上次备份改名另存，新的实验就能重新创建备份。

```bash
mv -n test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc.tutorial39.bak \
  "test/plugin/checker/framework/task_graph_generator_v3/task_graph_mem_conflict_v3_test.cc.tutorial39.bak.$(date +%Y%m%d_%H%M%S)"
```

在完整通信算法中，类似依赖通常由流内顺序、Record/Wait 或 CCU 同步操作建立。定位时把日志中的任务编号和地址对应回相关同步代码，再修改、重编译、重跑原用例。设备侧算法的重新打包方式见本节第 7 小节。

### 修改源码后，更新实际使用的编译产物

**执行位置：Ubuntu 终端**（若还在 `(hvm)$>` 中，先输入 `exit` 返回）。

首次跑通以后，如果修改了源码，先在普通 Linux 终端中重新设置路径：

```bash
cd /home/workspace/hcomm/test/hccl_vm

source /home/workspace/Ascend/cann/set_env.sh
export HCCL_CODE_HOME=/home/workspace/hccl
export HCOMM_CODE_HOME=/home/workspace/hcomm
```

然后按修改内容，选择对应命令：

修改 HCCL 源码时：

```bash
bash build_pkg.sh --install hccl
```

修改 HCOMM 源码时：

```bash
bash build_pkg.sh --install hcomm
```

修改两个仓库的源码时：

```bash
bash build_pkg.sh --full
```

修改 HCCL-VM 工具自身源码时：

```bash
bash build.sh --full
```

这里最容易混淆的是：

- **`build.sh --full`：构建 HCCL-VM 工具。**
- **`build_pkg.sh`：仅部署现有 CANN 包中的设备侧依赖。**
- **`build_pkg.sh --install hccl`：构建并安装 HCCL，再部署它的设备侧依赖；首次检查发现主机端 HCCL 文件缺失时也使用这条命令。**
- **`build_pkg.sh --install hcomm`：构建并安装 HCOMM，再部署它的设备侧依赖。**
- **`build_pkg.sh --full`：构建并安装 HCCL、HCOMM，再更新模拟器所需的设备侧库。**

改完源码后必须更新实际使用的编译产物，再按 3.5 节 重新启动环境跑测试，修改才会生效。[重新构建与部署脚本](https://gitcode.com/cann/hcomm/blob/master/test/hccl_vm/build_pkg.sh)

### 其他执行模式

**执行位置：Ubuntu 终端**（若还在 `(hvm)$>` 中，先输入 `exit` 返回）。

CCU、AICPU、HostDPU、AIV 是不同的通信任务展开 / 执行模式。

| 模式 | 配置或区别 |
|---|---|
| CCU | `HCCL_OP_EXPANSION_MODE="CCU_SCHED"`，本指南首先跑通的模式 |
| AICPU | `HCCL_OP_EXPANSION_MODE="AI_CPU"`，通过 QEMU 模拟 ARM 设备侧程序 |
| HostDPU | 环境变量与 AICPU 相同，但使用专用的 HostDPU 集群拓扑，通信域至少包含两台模拟服务器 |
| AIV | `HCCL_OP_EXPANSION_MODE="AIV"`，选择 AIV 模式 |

切换模式时，应在普通 Linux 终端中修改环境变量，再启动新的模拟会话。HostDPU 使用的启动配置是 `ascend950_cluster_32_server_normal_hostdpu.yaml`，不能直接沿用单服务器的 `112` 通信域。[执行模式说明](https://gitcode.com/cann/hcomm/tree/master/test/hccl_vm)

改 `hccl_test` 的源码时，重新 `make` 之后产物直接落在
`$ASCEND_HOME_PATH/tools/hccl_test/bin`，不需要额外部署。

原课件里"改错一条任务依赖 → 读 Checker 的定位信息 → 恢复源码"的完整练习，
就是在这个本地环境里做的：改 `task_graph_generator_v3` 下的生成器源码，
重新 `make`，再跑同一个测试对比 Checker 的报错。

## 7. 打包成绿色包

本地环境跑通之后，可以打成包给云端用。**三个坑必须避开**（都是 HCCL 的硬编码路径，`ldd` 看不见）：

1. **库目录必须叫 `lib64/`**，不能叫 `libs/`——HCCL 按 `$ASCEND_HOME_PATH/lib64/...` 找；
2. **`lib64/hccl_aiv_op_910_95.o`**（1.4 MB）——AIV 算子二进制；
3. **`opp/built-in/op_impl/aicpu/{config,kernel}/`**——缺 `config/ccl_kernel.json` 会直接
   `HcclCommInitClusterInfoV2 failed`。

再加两样：`conf/error_manager/error_code.json`、以及启动脚本里的
`export ASCEND_HOME_PATH="$HERE"`。

> 排查缺失文件的方法：不要只 grep `*.so`，HCCL 还读 `.o`、`.json`、`version.info`。
> 直接从日志的 `[error]` 行里提取绝对路径，逐个比对包里有没有。

## 8. 常见问题

| 现象 | 原因 | 处理 |
| --- | --- | --- |
| 编译报 `ascend_hal.h` 找不到 | CANN 版本太老 | 升级 CANN 后重新编译 |
| 编译报 `HcclCommSymWindow` 未声明 | 同上 | 升级 CANN 后重新编译 |
| `make` 被杀（OOM） | `-j` 开太大 | 改 `-j6` 或 `-j3` |
| `make install` 报没有 install 目标 | 只跑了 `BUILD_DEVICE_ARM=ON` | 先跑 `=OFF` 那一遍 |
| 缺 `libhccl.so` / `hccl.h` | 只装了 toolkit | 补装 950-ops |
| `hccl_test` 链接报 TaskService 未定义 | `--as-needed` 丢了 `-lhccl` | 用第 4 节的 `LIBS=` 覆盖 |

[← 3.8 问题定位](./03.08_debug_and_verify.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb)